# Day 3 · 실험 설계와 발표 — 기준 조건 위에서 질문하기
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kehdgus9434/sonic-vibe-coding-lab/blob/main/notebooks/day3_experiment.ipynb)

**오늘의 결과물**: 기준 실험 표 + 나만의 실험 1개(가설·지표·결과·한계) + 3분 발표

| 시간 | 내용 |
|---|---|
| 0:00–0:10 | 환경 준비 · 기준/통제 조건이란 |
| 0:10–0:35 | Mission 3-1 기준 실험 3조건 + 지표 추가 |
| 0:35–1:20 | Mission 3-2 자유 실험 |
| 1:20–1:30 | Mission 3-3 결과 영상 |
| 1:30–1:55 | 발표 (1인/1조 3분) |
| 1:55–2:00 | 마무리 체크리스트 |

In [ ]:
#@title 0. 실습 환경 준비 — 처음 한 번 실행 (약 1–2분)
REPO = "https://github.com/kehdgus9434/sonic-vibe-coding-lab"   # 에이전트형 도구 사용자는 내 Fork 주소로 바꾸세요
import os, sys, subprocess
from pathlib import Path
if Path("/content").exists():                              # Google Colab
    LAB_DIR = Path("/content/sonic-vibe-coding-lab")
    if not LAB_DIR.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, str(LAB_DIR)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(LAB_DIR / "requirements-colab.txt")], check=True)
else:                                                      # 로컬 Jupyter (저장소 안에서 실행)
    LAB_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(LAB_DIR)
sys.path.insert(0, str(LAB_DIR))
import numpy as np
import sonic_lab
from sonic_lab import check, reference as ref, render, analysis
lab = sonic_lab.start()

### 📝 프롬프트 로그 (이 셀을 더블클릭해 편집)
| # | 내가 한 요청 | AI 답 요약 | 내가 확인한 근거 | 채택? |
|---|---|---|---|---|
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |

---
## Mission 3-1 · 기준 실험 3조건 (25분)
실제 연구(06→07번 노트북)에서 원인을 가려낸 방법입니다. 같은 샘플·같은 시뮬레이터에서 세 조건을 비교합니다.

| 조건 | 정책 입력 | 의미 |
|---|---|---|
| `reference_pd` | 없음 — 참조 관절각을 PD로 그대로 재생 | 정책 없이도 걸을 수 있나? |
| `0` (G1 모드) | G1 참조 관절각 | 로봇 모션을 추종 |
| `2` (SMPL 모드) | 사람 관절 위치 | 사람 모션을 추종 |

In [ ]:
N = 1000   # 20초. 시간이 남으면 None(전체 40초)
baseline = {f"cond:{m}": ref.run_physics(lab.sim, lab.policy, lab.motion, mode=m, max_frames=N) for m in ["reference_pd", 0, 2]}
display(analysis.table(baseline))
analysis.plot(baseline, lab);

> 🧭 **방법**: 아래 설명 + `docs/01_contract.md` + 함수 시그니처를 바이브 코딩 도구에 주고 ✍️ 셀을 채운 뒤, ✅ 셀로 확인합니다.

1. **해석**: `reference_pd`가 먼저 넘어지는 이유를 AI와 토론하고, 근거가 되는 수치(골반 높이 그래프 등)를 하나 찾으세요.
2. **지표 추가**: 관절 오차를 **부위별**(다리 12 / 허리 3 / 팔·손목 14, MuJoCo 순서 계약서 2절)로 나누는 함수를 바이브 코딩으로 만들고 모드 0과 2를 비교하세요.

In [ ]:
# ✍️ Mission 3-1 — 부위별 관절 오차 (deg)
def joint_error_by_group(arrays, lab):
    """반환 예: {"legs": 3.1, "waist": 1.2, "arms": 7.8}"""
    raise NotImplementedError

for label in ["cond:0", "cond:2"]:
    print(label, joint_error_by_group(baseline[label][1], lab))

**해석 메모**: (여기에 3문장)

---
## Mission 3-2 · 자유 실험 (45분)
아래에서 하나를 고르거나 직접 제안하세요. **실험 카드를 먼저 채우고** 코드를 작성합니다.

| 카드 | 질문 | 힌트 |
|---|---|---|
| A. 좌우 반전 | 반전 모션도 같은 품질로 추종할까? | `lab.load_motion(mirrored=True)` (공식 `_M` 샘플) |
| B. 외란 강건성 | 골반을 몇 N으로 밀면 넘어질까? | `run_physics(..., on_frame=fn)`에서 `sim.data.xfrc_applied[sim.pelvis_id, :3]` 설정. **힘을 끄는 것도 잊지 말 것** |
| C. 입력 잡음·지연 | 사람 관절 위치에 잡음 σ cm, 또는 k프레임 지연이 생기면? | `dataclasses.replace(lab.motion, local24=...)` |
| D. 재생 속도 | 0.8배/1.2배 속도 걷기도 따라갈까? | 시간축 재보간: 위치는 선형, 회전은 SLERP (계약서 1절) |
| E. 손목 정보 제거 | 손목 각도를 0으로 바꾸면(근사 데이터의 한계)? | `replace(lab.motion, wrists_il=np.zeros_like(...))` |
| F. 내 데이터 | 다른 동작 데이터도 될까? | 강사가 OpenSim NPZ를 주면 `ref.load_opensim_npz(path)` (근사 변환) |

`run_physics`는 `(summary, arrays)`를 돌려줍니다. 여러 조건을 `{"라벨": 결과}`로 모아 `analysis.table`, `analysis.plot`에 넣으세요.

In [ ]:
# 실험 카드 (먼저 채우기)
card = {
    "가설": "",
    "변인": "",
    "지표": "",
    "통제조건": "",
    "결과": "",
    "해석": "",
    "한계": "",
}

In [ ]:
# ✍️ Mission 3-2 — 실험 코드
import dataclasses
results = {"기준(mode 2)": ref.run_physics(lab.sim, lab.policy, lab.motion, mode=2, max_frames=N)}

# results["내 조건"] = ...

display(analysis.table(results))
analysis.plot(results, lab);

In [ ]:
# ✅ 카드 확인 (결과·해석·한계까지 채운 뒤)
check.experiment_card(card)

---
## Mission 3-3 · 결과 영상 (10분)
발표에 쓸 6초 영상을 만듭니다. 영상은 *보여주기*일 뿐, 판단은 표의 수치로 합니다.

In [ ]:
label = list(results)[-1]
video = render.mesh_video(lab, results[label][1], f"outputs/day3_{len(results)}.mp4", label=label, seconds=6)
render.show(video or render.skeleton_video(lab, results[label][1], "outputs/day3_skeleton.mp4", title=label, seconds=6))

---
## 발표 (3분) — `templates/presentation.md`
1. 질문과 가설 → 2. 기준 vs 내 조건 (표 1개 + 그림/영상 1개) → 3. 해석과 한계 → 4. 바이브 코딩 회고 1줄

## 마무리 — `docs/03_checklist.md`
내 결과에 대해 체크리스트를 채울 수 있나요? 채울 수 없는 항목이 다음 연구의 첫 번째 할 일입니다.